# 5.2 Family History and Lifestyle

**Survey series, class 2.** Type 2 diabetes runs in families, and is linked to habits like smoking and drinking. Health records rarely capture these, but the *All of Us* surveys do. In this class we turn three survey topics, **family history of diabetes**, **smoking**, and **alcohol**, into variables, and ask how they relate to type 2 diabetes. Along the way we meet the biggest trap in survey analysis: answers given **after** a diagnosis.

### By the end of this notebook you'll be able to

1. Code a **select all that apply** family history question into one variable per relative
2. Choose the right **denominator**: who could have answered a question at all
3. Compare **self-reported** diabetes with the health records
4. Check **when** a survey was answered relative to a diagnosis, and explain **reverse causation**
5. Fit age- and sex-adjusted **logistic regression** models with survey variables, and run a **sensitivity analysis**

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Are family history of diabetes, smoking, and alcohol use linked to type 2 diabetes? |
| 🧪 **The analysis** | Percents by group; **logistic regression** adjusted for age and sex; the same model using only cases who answered **before** their diagnosis |
| 🎯 **Why** | Family history and lifestyle are classic risk factors. Logistic regression compares the **odds** of diabetes, adjusted for age and sex (which differ between groups) |
| ✅ **What we conclude** | How strongly each factor is linked to type 2 diabetes in our sample, and how much the timing of answers matters |
| ⚠️ **What it can't tell us** | Cause. Most answers were given **after** diagnosis, so they can reflect the diagnosis (people quit smoking or drinking after it). Family history is reported, not verified |
| 🔗 **Connections** | Uses `survey_basics.csv` and `survey_questions.csv` from **5.1**; models from **3.2**; family history connects to the genetics in **4.2** (*TCF7L2*) |

| | |
|---|---|
| 📂 **Uses** | the *All of Us* Controlled Tier dataset; `cohort_sample.csv` (**1.1**), `survey_questions.csv` and `survey_basics.csv` (**5.1**) |
| 💾 **Saves** | `survey_lifestyle.csv` (one row per person, for **5.3**) |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Setup, and finding the alcohol questions | 5 |
| 3. When did people answer? | 10 |
| 4. Family history | 20 |
| 5. Alcohol | 10 |
| 6. Describe and model | 15 |
| 7. Before or after diagnosis? | 10 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 90 minutes |
| ⏱️ **Run time** | about **3 minutes** to run every cell |
| 💰 **Data cost** | about **$1**: two queries on `ds_survey` |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$1.50** per student for this class |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### Family history

Type 2 diabetes is more common in people whose **parents or siblings** have it. Family history reflects **shared genes** (like the *TCF7L2* variant in **4.2**) and a **shared environment** (diet, activity, income). The *All of Us* question:

> *Including yourself, who in your family has had type 2 diabetes? Select all that apply.*

The answers include *Self* and relatives (mother, father, sibling, …). So one question gives us **two** things: family history (the relatives), and **self-reported** diabetes (*Self*).

### The denominator: who could answer?

This question is in **Personal and Family Health History**, a later survey that only about half our sample answered (**5.1**). Someone who **never took the survey** has no row, but neither does someone who took it and had **no relative** with type 2 diabetes. So:

| Took the survey? | Row for this question? | Family history |
|---|---|---|
| no | no | **unknown** (`NA`) |
| yes | no | **no** relative reported |
| yes | a relative | **yes** |
| yes | only *PMI: Skip* (or similar) | **unknown** (`NA`) |

### Reverse causation

A survey answer describes the person **when they answered**. If someone was diagnosed with diabetes in 2010 and answered the Lifestyle survey in 2020, their answers can reflect the **diagnosis**: doctors advise people with diabetes to stop smoking and cut down on alcohol. This is **reverse causation**: the outcome affects the "risk factor", not the other way round.

A well-known example is the **sick quitter**: people who stop drinking because they're ill. If they're counted with lifelong non-drinkers, drinking can look **protective**. That's why we keep *never drank* and *none in the past year* apart.

### Confounding

A **confounder** is linked to both the "risk factor" and the outcome, and can create a link between them that isn't causal. Age is one (we adjust for it). Others, like **income** and **education**, are linked to drinking, smoking, **and** diabetes; they're the topic of **5.3**.

### Odds ratios (from 3.2)

| Odds ratio (OR) | Means |
|---|---|
| 1 | no difference in the odds of diabetes |
| above 1 | higher odds (e.g. 2 = twice the odds) |
| below 1 | lower odds |

Each OR is compared with a **reference** group (e.g. never smokers), and **adjusted** for the other variables in the model.

## Part 2. Setup

This class uses two files from **5.1**: the question dictionary (`survey_questions.csv`) and the smoking and general health variables (`survey_basics.csv`). If either is missing, run **5.1** first.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

cohort_sample <- load_cohort_sample()
questions     <- load_data("survey_questions.csv")
survey_basics <- load_data("survey_basics.csv", col_types = cols(person_id = "c"))

ids_sql <- str_c(cohort_sample$person_id, collapse = ", ")   # don't print: participant IDs

# From 5.1: drop the question's short name from an answer ("General Health: Excellent"
# becomes "Excellent"), but leave non-answers as they are ("PMI: Skip")
answer_text <- function(answer) {
  if_else(str_starts(answer, "PMI:"), answer, str_remove(answer, "^.*: "))
}

cohort_sample %>% count(cohort)

### Finding the alcohol questions by concept code

In 5.1 we searched the dictionary by **text**. Here we look questions up by their **concept code**, the stable name used in the codebooks. First, every alcohol question:

In [ ]:
questions %>%
  filter(str_detect(question, regex("alcohol|drink", ignore_case = TRUE))) %>%
  arrange(desc(n_people)) %>%
  select(survey, question_concept_id, concept_code, question, n_people)

**🔍 Read the output:** the Lifestyle alcohol questions. Like smoking, the first one (*Alcohol Participant*: have you **ever** had a drink?) opens follow-ups such as how often people drank in the **past year**: compare their `n_people`.

We use two of them, looked up by code:

In [ ]:
CODES <- c(alcohol_ever    = "Alcohol_AlcoholParticipant",
           drink_frequency = "Alcohol_DrinkFrequencyPastYear")

alcohol_questions <- questions %>%
  filter(concept_code %in% CODES) %>%
  mutate(name = names(CODES)[match(concept_code, CODES)])

alcohol_questions %>% select(name, question_concept_id, concept_code, question, n_people)

QUESTIONS <- c(family_t2d = 836800,
               setNames(alcohol_questions$question_concept_id, alcohol_questions$name))
QUESTIONS

**🔍 Read the output:** two rows, one per code, and `QUESTIONS` with three named IDs. If a row is missing, the code is spelled differently: find it in the table above and fix `CODES`.

## Part 3. When did people answer?

For each person: the date they first answered **Lifestyle** (smoking, alcohol) and **Personal and Family Health History** (family history). No date means they didn't take that survey.

💰 About $0.50.

In [ ]:
survey_dates <- run_query(str_glue("
  SELECT person_id, survey, MIN(DATE(survey_datetime)) AS first_date
  FROM `{CURRENT_DATASET}.ds_survey`
  WHERE survey IN ('Lifestyle', 'Personal and Family Health History')
    AND person_id IN ({ids_sql})
  GROUP BY person_id, survey")) %>%
  mutate(person_id = as.character(person_id),
         survey    = recode(survey, "Lifestyle" = "lifestyle_date",
                                    "Personal and Family Health History" = "pfhh_date")) %>%
  pivot_wider(names_from = survey, values_from = first_date)

people <- cohort_sample %>%
  select(person_id, cohort, age, sex_at_birth, first_t2d_date) %>%
  left_join(survey_dates, by = "person_id")

people %>%
  group_by(cohort) %>%
  summarise(people         = n(),
            pct_lifestyle  = round(100 * mean(!is.na(lifestyle_date)), 1),
            pct_pfhh       = round(100 * mean(!is.na(pfhh_date)), 1))

**🔍 Read the output:** almost everyone answered Lifestyle (99% of cases, 98% of controls in our test run); about half answered Personal and Family Health History, fewer cases (49%) than controls (57%), as in **5.1**.

### Before or after the diagnosis?

For **cases**, compare each survey date with the first type 2 diabetes date in their health records (`first_t2d_date`, from the cohort query):

In [ ]:
people %>%
  filter(cohort == "case") %>%
  summarise(cases_with_lifestyle       = sum(!is.na(lifestyle_date)),
            pct_answered_after_dx      = round(100 * mean(lifestyle_date > first_t2d_date, na.rm = TRUE), 1),
            median_years_after_dx      = round(median(as.numeric(lifestyle_date - first_t2d_date) / 365.25, na.rm = TRUE), 1))

**🔍 Read the output:** most cases answered the Lifestyle survey **after** their diabetes was first recorded: in our test run, 77% of cases, a median of 2.6 years after. So for most cases, "do you smoke now?" and "how often did you drink in the past year?" describe life **with** diabetes, not before it. Keep this in mind for every result below; Part 7 tests how much it matters.

## Part 4. Family history

💰 About $0.50: the family history and alcohol answers for our sample.

In [ ]:
answers <- run_query(str_glue("
  SELECT person_id, question_concept_id, answer, DATE(survey_datetime) AS survey_date
  FROM `{CURRENT_DATASET}.ds_survey`
  WHERE question_concept_id IN ({str_c(QUESTIONS, collapse = ', ')})
    AND person_id IN ({ids_sql})")) %>%
  mutate(person_id = as.character(person_id),
         name      = names(QUESTIONS)[match(question_concept_id, QUESTIONS)])

answers %>% count(name)

### Every answer to the family history question

In [ ]:
family <- answers %>% filter(name == "family_t2d")

family %>% count(answer, sort = TRUE) %>% hide_small_counts()

**🔍 Read the output:** each answer repeats the question, then names one family member after ` - `: *Self*, *Mother*, *Father*, *Sibling*, *Grandparent*, *Daughter*, or *Son*. One person can have several rows.

Note **PMI: Skip**: in our test run, about 1,000 rows, one of the most common answers. What does a skip mean on a **select all that apply** question? Perhaps "I skipped it", but perhaps "**none** of these people": with nothing to select, moving on may be recorded as a skip. We treat it as unknown (`NA`) for now, and test the other choice in Part 7.

### Group the relatives

First we keep only the **family member** from each answer: the text after the last `?`, `-`, or `:`. This matters: the question itself says "Including your**self**", so a search for *self* in the full answer would match **every** answer.

Then we group the members with `case_when()`. The **order** of the rules matters: *Grandmother* contains *mother*, so grandparents must be checked **before** parents. Then we **check** the grouping, member by member.

In [ ]:
family <- family %>%
  mutate(member   = if_else(str_starts(answer, "PMI:"), answer,
                            str_trim(str_extract(answer, "[^?:-]+$"))),   # the text after the last ?, -, or :
         relative = case_when(
           str_starts(answer, "PMI:")                                              ~ "non-answer",
           str_detect(member, regex("^self$", ignore_case = TRUE))                 ~ "self",
           str_detect(member, regex("grand", ignore_case = TRUE))                  ~ "grandparent",
           str_detect(member, regex("mother|father|parent", ignore_case = TRUE))   ~ "parent",
           str_detect(member, regex("sibling|brother|sister", ignore_case = TRUE)) ~ "sibling",
           str_detect(member, regex("daughter|\\bson\\b|child", ignore_case = TRUE)) ~ "child",
           TRUE                                                                    ~ "other"))

family %>% count(relative, member) %>% hide_small_counts()

**🔍 Read the output:** every family member, with the group it was put in. Check each row makes sense. Anything in `other` needs a look: add a rule for it if it's a relative we want.

### One row per person

`distinct()` keeps one row per person per group (someone who answered *Mother* and *Father* is one `parent`). Then `pivot_wider()` (from **2.2**) makes one `TRUE`/`FALSE` column per group.

Then the **denominator** from Part 1: everyone who took the survey. People who took it but have no row get `FALSE` (no relative reported). People who gave **only** non-answers, or never took the survey, get `NA`.

In [ ]:
groups <- c("self", "parent", "sibling", "child", "grandparent", "other")

family_wide <- family %>%
  filter(relative != "non-answer") %>%
  distinct(person_id, relative) %>%
  mutate(value = TRUE) %>%
  pivot_wider(names_from = relative, values_from = value, values_fill = FALSE)
for (g in setdiff(groups, names(family_wide))) family_wide[[g]] <- FALSE   # groups nobody chose

only_nonanswers <- setdiff(family$person_id, family_wide$person_id)

family_history <- people %>%
  filter(!is.na(pfhh_date)) %>%                                  # took the survey
  select(person_id) %>%
  left_join(family_wide, by = "person_id") %>%
  mutate(across(all_of(groups), ~ replace_na(.x, FALSE)),        # no row: no relative reported
         family_history = parent | sibling,                      # first-degree relatives (not children)
         across(c(all_of(groups), family_history),
                ~ if_else(person_id %in% only_nonanswers, NA, .x)))

nrow(family_history)   # people who took the survey
family_history %>% count(family_history)

**🔍 Read the output:** one row per person who took the survey. `family_history` is `TRUE` if a **parent or sibling** had type 2 diabetes. In our test run: about 1,500 `TRUE`, 2,800 `FALSE`, and 1,000 `NA` (only a skip).

Why parents and siblings, but not children? A child's diabetes usually comes **after** the parent's, so it can't explain the parent's risk; it also depends on how old your children are. Grandparents are often left out because people know less about their health.

### Self-report vs the health records

Our cases were chosen from **health records**. How many said *Self*?

In [ ]:
self_report <- people %>%
  inner_join(family_history %>% select(person_id, self), by = "person_id") %>%
  filter(!is.na(self)) %>%
  count(cohort, self) %>%
  hide_small_counts() %>%
  group_by(cohort) %>%
  mutate(percent = round(100 * n / sum(n, na.rm = TRUE), 1)) %>%
  ungroup()

self_report

**🔍 Read the output:**

- **Cases** who said *Self*: the share whose diagnosis in the records is also reported in the survey. In our test run, only **59%**. Some answered **before** they were diagnosed; some reported *Self* for prediabetes or "other diabetes" instead; some records may be wrong (e.g. a single type 2 code, **1.1**).
- **Controls** who said *Self*: people with **no** diabetes code in their records who report having type 2 diabetes. Only about **2%** in our test run. They may have been diagnosed somewhere whose records *All of Us* doesn't have.

Neither source is perfect. Agreement between them is a good check of any case definition.

**💬 Discuss:** should controls who report having type 2 diabetes stay controls? (For a sensitivity analysis, remove them and see if the results change.)

### Family history in cases and controls

In [ ]:
people %>%
  inner_join(family_history, by = "person_id") %>%
  filter(!is.na(family_history)) %>%
  group_by(cohort) %>%
  summarise(people             = n(),
            pct_parent         = round(100 * mean(parent), 1),
            pct_sibling        = round(100 * mean(sibling), 1),
            pct_family_history = round(100 * mean(family_history), 1))

**🔍 Read the output:** family history of type 2 diabetes is much more common in cases: in our test run, **45%** of cases and **26%** of controls had a parent or sibling with it (parent 36% vs 22%; sibling 21% vs 8%). Part 6 adjusts for age and sex.

### ✍️ Your turn

Is a **mother** with type 2 diabetes reported more often than a **father**? Count the two answers separately, among people who took the survey.

<details>
<summary><b>Show answer</b></summary>

```r
family %>%
  mutate(who = case_when(str_detect(member, regex("grand", ignore_case = TRUE))  ~ NA_character_,
                         str_detect(member, regex("mother", ignore_case = TRUE)) ~ "mother",
                         str_detect(member, regex("father", ignore_case = TRUE)) ~ "father")) %>%
  filter(!is.na(who)) %>%
  distinct(person_id, who) %>%
  count(who) %>%
  hide_small_counts()
```

Mothers are usually reported **more** often. One reason is that people often know more about their mother's health than their father's: an example of how **reporting** shapes survey data.

</details>

In [ ]:
# Your code here

## Part 5. Alcohol

Alcohol works like smoking in 5.1: the first question (*ever had a drink?*) opens the follow-up (*how often in the past year?*). First, every combination:

In [ ]:
alcohol <- answers %>%
  filter(name %in% c("alcohol_ever", "drink_frequency")) %>%
  mutate(answer = answer_text(answer)) %>%
  distinct(person_id, name, .keep_all = TRUE) %>%
  select(person_id, name, answer) %>%
  pivot_wider(names_from = name, values_from = answer)

alcohol %>% count(alcohol_ever, drink_frequency) %>% hide_small_counts() %>% print(n = 40)

**🔍 Read the output:** *No* to ever drinking → no frequency answer (never asked). *Yes* → a frequency for the past year, from *Never* to *4 or More Per Week*. As with smoking (**5.1**), some people who **skipped** the first question answered the follow-up.

We make one variable. Note the two kinds of non-drinker: people who **never** drank, and people who drank once but **not in the past year** (the possible "sick quitters"). As with smoking, *None in past year* needs *Yes* first: a skip followed by *Never* could be either kind.

In [ ]:
alcohol <- alcohol %>%
  mutate(alcohol_use = case_when(
    str_detect(alcohol_ever, regex("^no$", ignore_case = TRUE))         ~ "Never drank",
    str_detect(alcohol_ever, regex("^yes$", ignore_case = TRUE)) &
      str_detect(drink_frequency, regex("never", ignore_case = TRUE))   ~ "None in past year",
    str_detect(drink_frequency, regex("monthly", ignore_case = TRUE))   ~ "Monthly or less",
    str_detect(drink_frequency, regex("month", ignore_case = TRUE))     ~ "2-4 times a month",
    str_detect(drink_frequency, regex("week", ignore_case = TRUE))      ~ "2+ times a week",
    TRUE                                                                ~ NA_character_),
    alcohol_use = factor(alcohol_use, levels = c("Never drank", "None in past year", "Monthly or less",
                                                 "2-4 times a month", "2+ times a week")))

alcohol %>% count(drink_frequency, alcohol_use) %>% hide_small_counts()

**🔍 Read the output:** each frequency answer and the group it went into. Check that none landed in the wrong group (e.g. a *week* answer in *month*), and that only non-answers are `NA`.

## Part 6. Describe and model

One row per person, with everything together. `sex_at_birth` is limited to *Female* and *Male* for the models (the other answers are too rare to estimate), and `age10` is age in **10-year** units, so its odds ratio is "per 10 years".

In [ ]:
lifestyle <- people %>%
  left_join(survey_basics %>% select(person_id, smoking_status), by = "person_id") %>%
  left_join(alcohol %>% select(person_id, alcohol_use),          by = "person_id") %>%
  left_join(family_history %>% select(person_id, self, family_history), by = "person_id") %>%
  mutate(t2d            = as.integer(cohort == "case"),
         age10          = age / 10,
         smoking_status = factor(smoking_status, levels = c("Never", "Former", "Current")))

nrow(lifestyle)   # still 10,000
save_data(lifestyle %>% select(person_id, smoking_status, alcohol_use, self, family_history,
                               lifestyle_date, pfhh_date),
          "survey_lifestyle.csv")

### Percents in each group

In [ ]:
describe <- function(data, var) {
  data %>%
    filter(!is.na(.data[[var]])) %>%
    count(cohort, level = as.character(.data[[var]])) %>%
    hide_small_counts() %>%
    group_by(cohort) %>%
    mutate(percent = round(100 * n / sum(n, na.rm = TRUE), 1)) %>%
    ungroup() %>%
    select(-n) %>%
    pivot_wider(names_from = cohort, values_from = percent) %>%
    mutate(variable = var, .before = 1)
}

map(c("smoking_status", "alcohol_use", "family_history"), ~ describe(lifestyle, .x)) %>% list_rbind()

**🔍 Read the output:** the percent of cases and of controls in each group, among people who answered. Look for:

- **Smoking:** more **former** smokers among cases (31% vs 24% in our test run).
- **Alcohol:** cases less often drink **2+ times a week** (14% vs 27%), and more often had **none in the past year** (24% vs 15%).
- **Family history:** much more common in cases (45% vs 26%).

These are **unadjusted**: cases are older (**3.1**), and age is linked to all three.

### Logistic regression

Two models, because they use different people:

1. **Smoking and alcohol**: everyone who answered Lifestyle (almost the whole sample)
2. **Family history**: only people who took Personal and Family Health History (about half)

`glm()` leaves out anyone with a missing value; `nobs()` shows how many were used.

In [ ]:
model_data <- lifestyle %>% filter(sex_at_birth %in% c("Female", "Male"))

m_lifestyle <- glm(t2d ~ age10 + sex_at_birth + smoking_status + alcohol_use,
                   family = binomial, data = model_data)
m_family    <- glm(t2d ~ age10 + sex_at_birth + family_history,
                   family = binomial, data = model_data)

c(lifestyle_model = nobs(m_lifestyle), family_model = nobs(m_family))

odds_ratios <- bind_rows(
  broom::tidy(m_lifestyle, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "smoking and alcohol"),
  broom::tidy(m_family,    exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "family history")) %>%
  filter(term != "(Intercept)") %>%
  transmute(model, term,
            OR      = round(estimate, 2),
            CI_low  = round(conf.low, 2),
            CI_high = round(conf.high, 2),
            p_value = format.pval(p.value, digits = 2, eps = 0.001))

odds_ratios

**🔍 Read the output:** one odds ratio per variable, adjusted for the others in its model. Reference groups: *Never* smokers, *Never drank*, no family history, *Female*.

In our test run:

- **Family history:** OR **2.47** (2.16 to 2.83): about two and a half times the odds of type 2 diabetes, the largest odds ratio. Family history is a strong, well-known risk factor.
- **Smoking:** former smokers OR 1.24, current smokers 1.32, compared with never smokers.
- **Alcohol:** the more often people drank, the **lower** their odds: OR 0.74 for monthly or less, down to **0.26** for 2+ times a week, compared with people who never drank. *None in past year* was 0.87, close to never drinkers.

Does alcohol prevent diabetes? **This analysis can't say.** People with diabetes are advised to drink less (**reverse causation**, Part 7); people in poor health drink less; and people who drink more often tend to have **higher incomes** and more education, which are linked to lower diabetes risk (**confounding**, **5.3**).

### A forest plot

A **forest plot** shows each odds ratio as a point, with its 95% confidence interval as a line. The dashed line at 1 means "no difference"; the x-axis is on a **log scale**, so an OR of 2 and of 0.5 are the same distance from 1.

In [ ]:
odds_ratios %>%
  filter(!term %in% c("age10", "sex_at_birthMale")) %>%
  mutate(term = case_when(str_starts(term, "smoking_status") ~ str_c("Smoking: ", str_remove(term, "smoking_status")),
                          str_starts(term, "alcohol_use")    ~ str_c("Alcohol: ", str_remove(term, "alcohol_use")),
                          TRUE                               ~ "Family history (parent or sibling)"),
         term = fct_inorder(term)) %>%
  ggplot(aes(x = OR, y = fct_rev(term), colour = model)) +
  geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +
  geom_errorbar(aes(xmin = CI_low, xmax = CI_high), width = 0.2) +
  geom_point(size = 3) +
  scale_x_log10() +
  labs(x = "Odds ratio (log scale)", y = NULL, colour = "Model",
       title = "Family history and lifestyle",
       subtitle = "Odds ratios for type 2 diabetes, adjusted for age and sex") +
  theme_minimal(base_size = 14) +
  theme(legend.position = "bottom")

### ✍️ Your turn

Does the family history odds ratio change if you also adjust for **smoking** and **alcohol**? Fit the model and compare.

<details>
<summary><b>Show answer</b></summary>

```r
m_both <- glm(t2d ~ age10 + sex_at_birth + family_history + smoking_status + alcohol_use,
              family = binomial, data = model_data)
broom::tidy(m_both, exponentiate = TRUE, conf.int = TRUE) %>% filter(term == "family_historyTRUE")
```

If it barely changes, the family history link isn't explained by smoking or drinking. (The model uses only people with **all** the variables, so `nobs(m_both)` is a little smaller.)

</details>

In [ ]:
# Your code here

## Part 7. Before or after diagnosis?

If reverse causation explains part of the smoking and alcohol results, they should **change** when we use only cases who answered **before** their diagnosis: their answers can't reflect it.

We keep all controls, and only the cases whose Lifestyle date is **before** their first type 2 diabetes date. These cases were diagnosed **after** joining *All of Us*.

In [ ]:
before_data <- model_data %>%
  filter(cohort == "control" | lifestyle_date < first_t2d_date)

before_data %>% count(cohort)

m_before <- glm(t2d ~ age10 + sex_at_birth + smoking_status + alcohol_use,
                family = binomial, data = before_data)

bind_rows(
  broom::tidy(m_lifestyle, exponentiate = TRUE, conf.int = TRUE) %>% mutate(cases = "all cases"),
  broom::tidy(m_before,    exponentiate = TRUE, conf.int = TRUE) %>% mutate(cases = "answered before diagnosis")) %>%
  filter(str_detect(term, "smoking|alcohol")) %>%
  transmute(term, cases, OR = sprintf("%.2f (%.2f to %.2f)", estimate, conf.low, conf.high)) %>%
  pivot_wider(names_from = cases, values_from = OR)

**🔍 Read the output:** far fewer cases (about 1,100 in our test run, only those diagnosed after they answered), so wider confidence intervals. Compare the two columns:

- If an odds ratio **changes** with cases who answered before diagnosis, part of the original result was probably **reverse causation**.
- If it stays about the same, the link was there **before** the diagnosis.

In our test run:

- **Current smoking** rose from 1.32 to **1.71**. That's what reverse causation predicts: after a diagnosis, some smokers quit, so in later answers fewer cases are current smokers, which **hides** part of the link.
- **Alcohol** barely changed: 2+ times a week was 0.26 with all cases and **0.31** with cases who answered before diagnosis. So the lower odds in drinkers is **not** mainly people cutting down after their diagnosis. Confounding (income, education, health) remains a likely explanation: **5.3** looks at it.

This is a **sensitivity analysis**: rerunning an analysis with a different, defensible choice, to see whether the conclusion holds.

### Another choice: what does a skip mean?

In Part 4, people whose only answer to the family history question was *PMI: Skip* got `NA`. What if a skip really meant "no one"? Code them as `FALSE` and refit:

In [ ]:
model_data <- model_data %>%
  mutate(family_history_skip_no = if_else(!is.na(pfhh_date) & is.na(family_history), FALSE, family_history))

model_data %>% count(cohort, family_history_skip_no) %>% hide_small_counts()

m_skip_no <- glm(t2d ~ age10 + sex_at_birth + family_history_skip_no, family = binomial, data = model_data)

bind_rows(
  broom::tidy(m_family,  exponentiate = TRUE, conf.int = TRUE) %>% mutate(skip = "skip = unknown (NA)"),
  broom::tidy(m_skip_no, exponentiate = TRUE, conf.int = TRUE) %>% mutate(skip = "skip = no family history")) %>%
  filter(str_starts(term, "family_history")) %>%
  transmute(skip, people = c(nobs(m_family), nobs(m_skip_no)),
            OR = sprintf("%.2f (%.2f to %.2f)", estimate, conf.low, conf.high))

**🔍 Read the output:** the same model, with the skips counted as "no family history". More people are included. If the odds ratio barely moves, the choice doesn't matter much for the conclusion; if it moves a lot, report **both**, and say which you chose and why.

In our test run: 4,242 → 5,237 people, and the odds ratio rose from **2.47** to **2.95**. Controls skipped more often than cases, so counting skips as "no" adds mostly controls **without** family history, which widens the gap. The conclusion holds either way (a strong link), but its **size** depends on a coding choice: report both.

**💬 Discuss:** the "before" cases were diagnosed **after** joining *All of Us*. How might they differ from cases diagnosed long ago? (Younger at diagnosis? Diagnosed by routine screening?)

## ⚠️ Common errors

| Message or problem | Means | Fix |
|---|---|---|
| *'survey_questions.csv' does not exist* | 5.1 hasn't been run | Run 5.1 (Parts 2 and 4 are enough for the dictionary; all of it for `survey_basics.csv`) |
| `alcohol_questions` has fewer than 2 rows | a concept code is spelled differently | find the right code in the alcohol table, and fix `CODES` |
| A relative lands in `other` | a rule doesn't match its text | add a rule to `case_when()` (check the order!) |
| `glm()` uses far fewer people than expected | someone has a missing value in **any** variable | check `nobs()`, and which variables are missing |
| *glm.fit: fitted probabilities numerically 0 or 1* | a group where (almost) everyone is a case or a control | combine small groups |

## ✅ Key takeaways

- A select-all family history question becomes **one column per relative**, after checking how each answer was grouped
- Choose the **denominator** carefully: no row can mean "no", or "never asked"
- **Self-reported** diabetes and the records mostly agree, but not always: a useful check of any case definition
- Most survey answers came **after** diagnosis: watch for **reverse causation**, and for **confounding** by things like income
- A *PMI: Skip* on a select-all question is ambiguous: test both codings
- A **sensitivity analysis** (only answers given before diagnosis) shows how much timing matters

**Next:** **5.3 social_factors**: income, education, food security, and health care access, and type 2 diabetes.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).

## 🧑‍🏫 Instructor notes

- **Timing:** Part 4 (family history) takes longest. If short of time, skip the mother/father exercise.
- **The alcohol result** always prompts questions ("so drinking prevents diabetes?"). Ask students for explanations **before** Part 7: reverse causation, sick quitters, confounding by income and health. Part 7 rules out the first as the main one; 5.3 looks at income.
- **Family history and genes:** link back to 4.2. Family history captures far more than one variant: many genes, and a shared household.
- **Likely question:** "Why not use everyone for the family history model?" Because people who didn't take the survey aren't "no family history": they're unknown. Coding them as `FALSE` would make family history look rarer, and weaker.